In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression


In [ ]:
df = pd.read_csv(r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\data\processed\train_features.csv")

# Fix datetime
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
# Define features and target variable 
X = df.drop(['Weekly_Sales', 'Date'], axis=1)  # Date is excluded as the date is already extracted into features like month, week, etc.
y = df['Weekly_Sales']

In [ ]:
# DEfine feature type 
categorical_features = ['Type']
numerical_features = [col for col in X.columns if col not in categorical_features]

In [ ]:
# DEfine the preprocessor 
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ]
)

In [ ]:
# BUild the pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
# Train and Test split 
# Sort by Date first
df = df.sort_values('Date')

# 80/20 split index
split_index = int(len(df) * 0.8)

# Split dataframe
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Create X and y
X_train = train_df.drop(['Weekly_Sales', 'Date'], axis=1)
y_train = train_df['Weekly_Sales']

X_test = test_df.drop(['Weekly_Sales', 'Date'], axis=1)
y_test = test_df['Weekly_Sales']

print(X_train.shape, X_test.shape)

In [ ]:

print(X_train.columns)

# MODEL TRAINING AND COMPARISON 


In [ ]:
# define the mdel with same preprocessor and regressor 

#linear regression (baseline)
from sklearn.linear_model import LinearRegression

model_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])



In [ ]:
#decision tree 
from sklearn.tree import DecisionTreeRegressor

model_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(random_state=42))
])

In [ ]:
# Random forest 
from sklearn.ensemble import RandomForestRegressor


# Model tuning 
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# Evaluation metrics 

from sklearn.metrics import mean_squared_error
import numpy as np

def evaluate(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    return mse, rmse


In [ ]:


results = {}

results['Linear Regression'] = evaluate(model_lr)
results['Decision Tree'] = evaluate(model_dt)
results['Random Forest'] = evaluate(model_rf)

print(results)

In [ ]:
results['Random Forest Tuned'] = evaluate(model_rf)
print(results)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model_hgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(max_iter=100))
])


results['HistGB'] = evaluate(model_hgb)
print(results)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Predict on validation set
y_pred = model_hgb.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("HistGradientBoosting Performance")
print("R2 Score :", round(r2, 4))
print("MAE      :", round(mae, 2))
print("MSE      :", round(mse, 2))
print("RMSE     :", round(rmse, 2))

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

pred = model_rf.predict(X_test)

print("R2 :", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

In [ ]:
from xgboost import XGBRegressor

model_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

results['XGBoost'] = evaluate(model_xgb)
print(results)

# ==============================
# TEST DATA PIPELINE
# ==============================

In [ ]:
# load the test data 

import pandas as pd 

test = pd.read_csv(r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\data\raw\test.csv")
stores= pd.read_csv(r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\data\raw\stores.csv")
features = pd.read_csv(r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\data\raw\features.csv")


In [ ]:
# merge the data sames as train

test = pd.merge(test, features, on=['Store', 'Date'], how='left')
test = pd.merge(test, stores, on='Store', how='left')
print(test.columns)

In [ ]:
# Fix the isholiday 
test['IsHoliday'] = test['IsHoliday_x']
test.drop(['IsHoliday_x', 'IsHoliday_y'], axis=1, inplace=True)


In [ ]:
print(test.columns)

In [ ]:
# fill markdown 
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
test[markdown_cols] = test[markdown_cols].fillna(0)

In [ ]:
# DAte feature 
test['Date'] = pd.to_datetime(test['Date'])

test['Year'] = test['Date'].dt.year
test['Month'] = test['Date'].dt.month
test['Week'] = test['Date'].dt.isocalendar().week
test['DayOfWeek'] = test['Date'].dt.dayofweek

In [ ]:
# The lag and rolling features set to 0 as there is no weekly sales data # Check this later 
test['Lag_1'] = 0
test['Lag_2'] = 0
test['Lag_4'] = 0

test['Rolling_Mean_4'] = 0
test['Rolling_Std_4'] = 0

In [ ]:
# Final model prediction - model used is random forest 
X_test_final = test.drop(['Date'], axis=1)
# Ensure test columns match training columns exactly
X_test_final = X_test_final[X.columns]

model_rf.fit(X, y)

predictions = model_rf.predict(X_test_final)

In [ ]:
# Fit final model first
model_rf.fit(X_train, y_train)

# Predict on validation set
y_pred = model_rf.predict(X_test)

# Comparison dataframe
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison["Error"] = comparison["Actual"] - comparison["Predicted"]
comparison["Abs_Error"] = comparison["Error"].abs()

print(comparison.head(20))

In [ ]:
print("Average Absolute Error:", comparison['Abs_Error'].mean())
print("Median Absolute Error:", comparison['Abs_Error'].median())

import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.scatter(comparison['Actual'], comparison['Predicted'], alpha=0.5)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
# Should still run this once check the smaplesubmiision.csv and then add in the raw folder 
import pandas as pd 
sample = pd.read_csv(r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\data\raw\sampleSubmission.csv")

sample['Weekly_Sales'] = predictions

sample.to_csv(r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\data\processed\final_submission.csv", index=False)

In [ ]:
# Saving the final model 

import joblib
model.fit(X_train, y_train)
joblib.dump(model_rf, r"C:\Users\shrey\OneDrive\Desktop\retail-demand-forecasting\models\model.pkl")